1. Create a single dataframe with the concatenation of all input csv files, adding a column called country
2. Extract all videos that have no tag.
3. For each channel, determine the total number of views
4. Save all rows with disabled comments and disabled ratings, or that have video_error_or_removed in a new dataframe called excluded, and remove those rows from the original dataframe.
5. Add a like_ratio column storing the ratio between the number of likes and of dislikes
6. Cluster the publish time into 10-minute intervals (e.g. from 02:20 to 02:30)
7. For each interval, determine the number of videos, average number of likes and of dislikes.
8. For each tag, determine the number of videos
Notice that tags contains a string with several tags.

9. Find the tags with the largest number of videos
10. For each (tag, country) pair, compute average ratio likes/dislikes
11. For each (trending_date, country) pair, the video with the largest number of views
12. Divide trending_date into three columns: year, month, day
13. For each (month, country) pair, the video with the largest number of views
14. Read all json files with the video categories
15. For each country, determine how many videos have a category that is not assignable.

In [1]:
import pandas as pd
import numpy as np
import json

### Point 1
Create a single dataframe with the concatenation of all input csv files, adding a column called country

In [ ]:
folder_path = r"C:\Users\kekko\Downloads\Python\Foundations of Computer Science\csv_Data"
countries = ["CA", "DE", "FR", "GB", "IN", "JP", "KR", "MX", "RU", "US"]

# We define a function called load_country that takes a country code as an argument.
def load_country(code):
    try:
        df = pd.read_csv(f"{folder_path}\\{code}videos.csv.zst", compression='zstd', encoding="utf-8", encoding_errors="ignore")
        df["country"] = code
        print(f"Uploaded: {code}")
        return df
    except FileNotFoundError:
        print(f"WARNING: Can't find: {code}videos.csv.zst")

# We use a list comprehension to loop through each country code in the countries list, call the load_country function for each code, and store the resulting DataFrames in a list called dfs. 
# We also check if the returned DataFrame is not None before adding it to the list.
dfs = [df for code in countries if (df := load_country(code)) is not None]

if dfs:
    YT_Trends = pd.concat(dfs, ignore_index=True)
    print(f"YT_Trends has a total of {len(YT_Trends)} rows and {len(YT_Trends.columns)} columns")
    print(f"Is the index unique? {YT_Trends.index.is_unique}")

Uploaded: CA
Uploaded: DE
Uploaded: FR
Uploaded: GB
Uploaded: IN
Uploaded: JP
Uploaded: KR
Uploaded: MX
Uploaded: RU
Uploaded: US
YT_Trends has a total of 375942 rows and 17 columns
Is the index unique? True


### Point 2
Extract all videos that have no tag.

In [ ]:
# We want to find out how many videos in YT_Trends do not have any tag.
no_tag=YT_Trends.loc[YT_Trends["tags"]=="[none]"]

print(f"{len(no_tag)} videos do not have any tag")
print(no_tag)

37698 videos do not have any tag
           video_id trending_date  \
41      JwboxqDylgg      17.14.11   
58      9B-q8h31Bpk      17.14.11   
78      1UE5Dq1rvUA      17.14.11   
86      pmJQ4KwliX4      17.14.11   
98      lHcXhBojpeQ      17.14.11   
...             ...           ...   
375815  VGykknw9eCM      18.14.06   
375819  fAIX12F6958      18.14.06   
375865  gS1DbvHHVH0      18.14.06   
375873  E4c7EE8_IX0      18.14.06   
375938  1h7KV2sjUWY      18.14.06   

                                                    title       channel_title  \
41      Canada Soccer's Women's National Team v USA In...       Canada Soccer   
58      John Oliver Tackles Louis C.K. And Donald Trum...            TV Shows   
78                Taylor Swift Perform Ready For It - SNL          Ken Reactz   
86      LATEST Q POSTS: ROTHSCHILDS, HOUSE OF SAUD, lL...        James Munder   
98                       三屆TVB視帝，拋棄10年青梅竹馬髮妻，為娶小三還不惜與母絕交！               明星百曉生   
...                                 

### Point 3
For each channel, determine the total number of views

In [ ]:
# We want to find out how many views each channel has in total.
# Counts how many times the channel appears therefore is not the solution
# I converted it into a dataframe and also renamed the column as "total_views" instead of the original "views"
pd.DataFrame(YT_Trends.groupby("channel_title").sum().loc[:,"views"].rename("total_views"))

,total_views
channel_title,
! 세상에 무슨일이,3942977
!!8時だよ面白ネタ大集合,50207
!BTS・TWICE まとめ,7310
!Los amorosos Virales¡,6069
!t Live,240038
...,...
５分でできる DIY,272607
ＢＡＳＨｔｖ,116292
Ｋのフィッシングちゃんねる,37165


### Point 4
Save all rows with disabled comments and disabled ratings, or that have video_error_or_removed in a new dataframe called excluded, and remove those rows from the original dataframe.

In [5]:
# Removing all rows where comments_disabled, ratings_disabled, or video_error_or_removed are True, and stores them in a new DataFrame called 'excluded'. 
# It then prints the number of rows removed and the new length of YT_Trends.
condition = ((YT_Trends["comments_disabled"] == True) & 
             (YT_Trends["ratings_disabled"] == True)) |(
                 YT_Trends["video_error_or_removed"] == True)

# Check if both 'comments_disabled' and 'ratings_disabled' are True, or if 'video_error_or_removed' is True.
excluded = YT_Trends[condition].copy()
YT_Trends = YT_Trends[~condition].copy()

print(
    f"Rows removed: {len(excluded)}\n"
    f"Current YT_Trends rows: {len(YT_Trends)}\n"
    f"Original rows: 375942"
)


Rows removed: 2620
Current YT_Trends rows: 373322
Original rows: 375942


### Point 5
Add a like_ratio column storing the ratio between the number of likes and of dislikes

In [6]:
# Calculate the like ratio and add it as a new column to YT_Trends. The like ratio is defined as the number of likes divided by the number of dislikes.
YT_Trends['like_ratio'] = (YT_Trends.loc[:,"likes"]/(YT_Trends.loc[:,"dislikes"].replace(0, np.nan)))

# Display the title and like ratio of the videos, sorted by like ratio in descending order.
YT_Trends.loc[:, ["title", "like_ratio"]].sort_values(by="like_ratio", ascending=False)

,title,like_ratio
331431,Alone in the Game | AT&T AUDIENCE Network,11688.000000
296366,[Dance Practice] 몬스타엑스 (MONSTA X) - DRAMARAMA,3550.750000
221150,[Dance Practice] 몬스타엑스 (MONSTA X) - DRAMARAMA,3550.500000
259304,[BT21] Meet BT21,3233.857143
113540,Swing - Rivage (Prod. Le Motel),3177.000000
...,...,...
358439,Camera Goes on Japanese Sushi Conveyor Belt Sh...,NaN
360544,ULTRA LIVE presents Ultra Music Festival 2018 ...,NaN
363543,Coachella 2018 LIVE Channel 1,NaN
363744,Coachella 2018 LIVE Channel 1,NaN


### Point 6
Cluster the publish time into 10-minute intervals (e.g. from 02:20 to 02:30)

In [7]:
# Group the videos by channel_title and calculate the total number of views for each channel. 
# Then, create a new DataFrame with the channel_title as the index and a column named total_views containing the sum of views for each channel.
YT_Trends["publish_time"] = pd.to_datetime(YT_Trends["publish_time"])
YT_Trends["time_clustering"] = YT_Trends["publish_time"].dt.floor("10min")

print(YT_Trends["time_clustering"])

0        2017-11-10 17:00:00+00:00
1        2017-11-13 17:00:00+00:00
2        2017-11-12 19:00:00+00:00
3        2017-11-12 18:00:00+00:00
4        2017-11-09 11:00:00+00:00
                    ...           
375937   2018-05-18 13:00:00+00:00
375938   2018-05-18 01:00:00+00:00
375939   2018-05-18 17:30:00+00:00
375940   2018-05-17 17:00:00+00:00
375941   2018-05-17 17:00:00+00:00
Name: time_clustering, Length: 373322, dtype: datetime64[ns, UTC]


### Point 7
For each interval, determine the number of videos, average number of likes and of dislikes.

In [8]:
# Group the videos by time_clustering and calculate the number of videos, the mean of likes, and the mean of dislikes for each interval.
interval_operations = YT_Trends.groupby("time_clustering").agg({
    "video_id": "count",  # number of videos
    "likes": "mean",      # like mean
    "dislikes": "mean"    # dislike mean
})

# Rename the columns for each arithmetical operation
interval_operations = interval_operations.rename(
    columns={"video_id": "num_videos", "likes": "avg_likes", "dislikes": "avg_dislikes"})

print(interval_operations)

                           num_videos    avg_likes  avg_dislikes
time_clustering                                                 
2006-07-23 08:20:00+00:00           1   459.000000      152.0000
2007-03-05 16:20:00+00:00           9   336.666667        2.0000
2007-06-25 06:50:00+00:00          12   579.833333       11.5000
2007-12-03 20:50:00+00:00          16   187.937500       15.6875
2008-01-07 21:20:00+00:00          10    99.900000        2.0000
...                               ...          ...           ...
2018-06-14 02:30:00+00:00           1   853.000000       77.0000
2018-06-14 03:00:00+00:00           2  2304.500000       31.5000
2018-06-14 03:20:00+00:00           1  1414.000000       28.0000
2018-06-14 03:30:00+00:00           1  8481.000000      252.0000
2018-06-14 03:40:00+00:00           1   374.000000       24.0000

[30397 rows x 3 columns]


### Point 8
For each tag, determine the number of videos Notice that tags contains a string with several tags.

In [9]:
# Count the number of videos for each tag. Note that the tags are stored as a string of tags separated by the "|" character.
different_tags_no_nan=YT_Trends["tags"].replace("[none]", np.nan).dropna().str.split("|").explode()
videos_tags=different_tags_no_nan.value_counts()
print(videos_tags)

tags
"funny"                   14933
"comedy"                  11962
"2018"                    11029
"news"                     5955
"music"                    5590
                          ...  
"愛護"                          1
"刈る"                          1
マヤ暦 音１ 白い魔法使い フローズンマリー        1
"安住アナ"                        1
"共同通信"                        1
Name: count, Length: 886934, dtype: int64


### Point 9
Find the tags with the largest number of videos

In [ ]:
# Display the 50 tags with the largest number of videos.
print("The 50 tags with the largest number of videos:\n",videos_tags.head(50))

The 50 tags with the largest number of videos:
 tags
"funny"            14933
"comedy"           11962
"2018"             11029
"news"              5955
"music"             5590
"2017"              5500
"video"             5391
"humor"             5036
"television"        4160
"show"              4131
"review"            4012
"Pop"               3899
"vlog"              3824
"interview"         3822
"live"              3761
"food"              3629
"comedian"          3561
"funny videos"      3556
"tv"                3374
"trailer"           3245
"movie"             3230
"funny video"       3211
"how to"            3189
"Comedy"            3036
"entertainment"     3028
"rap"               2967
"celebrities"       2881
"official"          2880
"celebrity"         2872
"new"               2850
"talk show"         2811
"fun"               2744
"jokes"             2739
"hollywood"         2702
"humour"            2633
"challenge"         2621
"reaction"          2556
"film"              25

### Point 10
For each (tag, country) pair, compute average ratio likes/dislikes

In [11]:
# We only select the columns that are useful to avoid computational errors
tags = YT_Trends[["country", "tags", "likes", "dislikes"]].copy()
tags_exploded = tags.assign(tags=tags["tags"].str.split("|")).explode("tags")
tags_grouped = tags_exploded.groupby(["country", "tags"])[["likes", "dislikes"]].sum()
tags_grouped["average_ratio"] = tags_grouped["likes"] / tags_grouped["dislikes"].replace(0, np.nan)
tags_grouped = tags_grouped.dropna(subset=["average_ratio"])

# We have to drop the NaN values because they are the result of the division by zero in the case in which the number of dislikes is 0.
print(f"Average like-dislike ratio for each tag and country pair sorted in descending order:")
tags_grouped["average_ratio"].sort_values(ascending=False)

Average like-dislike ratio for each tag and country pair sorted in descending order:


country  tags                 
RU       "Originals"              11688.0
         AT&T                     11688.0
         "DirectTV"               11688.0
         "U-Verse"                11688.0
         "AT&T AUDIENCE"          11688.0
                                   ...   
         "شبكة رؤية الإخبارية"        0.0
         "#Элджей"                    0.0
         "Эщкеренок"                  0.0
         "Эщкере"                     0.0
         "Эшкере"                     0.0
Name: average_ratio, Length: 1120369, dtype: float64

### Point 11
For each (trending_date, country) pair, the video with the largest number of views

In [ ]:
# Find the video with the largest number of views for each country and trending_date paired. 
# Display the title, number of views, trending_date, and country for each of these videos.
idx_largest_views = YT_Trends.groupby(
    ["trending_date", "country"])["views"].idxmax()

# We use the idxmax function to find the index of the video with the largest number of views for each country and trending_date pair.
print(f"The video with the largest number of views for each country and trending_date paired:")
YT_Trends.loc[idx_largest_views, ["title", "views", "trending_date", "country"]].set_index(["trending_date", "country"]).sort_index()

The video with the largest number of views for each country and trending_date paired:


title  \
trending_date country                                                      
17.01.12      CA       Marvel Studios' Avengers: Infinity War Officia...   
              DE       Marvel Studios' Avengers: Infinity War Officia...   
              FR                    Avengers: Infinity War Trailer Tease   
              GB               Luis Fonsi, Demi Lovato - Échame La Culpa   
              IN       Marvel Studios' Avengers: Infinity War Officia...   
...                                                                  ...   
18.31.05      JP                   Maroon 5 - Girls Like You ft. Cardi B   
              KR                          [MV] AOA _ Bingle Bangle(빙글뱅글)   
              MX       Cardi B, Bad Bunny & J Balvin - I Like It [Off...   
              RU       [BadComedian] - Движение Вверх (Плагиат или ве...   
              US       Childish Gambino - This Is America (Official V...   

                           views  
trending_date country             
17.01.12      CA        56367282  
              DE        56367282  
              FR         7281189  
              GB       143408235  
              IN        56367282  
...                          ...  
18.31.05      JP         3057987  
              KR         4150448  
              MX        20723565  
              RU         3125598  
              US       217750076  

[1967 rows x 2 columns]

### Point 12
Divide trending_date into three columns: year, month, day

In [ ]:
# We want to find out how many videos are trending for each day of the week.
YT_Trends["trending_date"] = pd.to_datetime(YT_Trends["trending_date"], format='%y.%d.%m')
YT_Trends['Day'] = YT_Trends["trending_date"].dt.day
YT_Trends['MonthNumber'] = YT_Trends["trending_date"].dt.month
YT_Trends['Month'] = YT_Trends["trending_date"].dt.month_name()
YT_Trends['Year'] = YT_Trends["trending_date"].dt.year

# Display the last 4 columns of YT_Trends to check if the new columns have been added correctly.
print(YT_Trends.iloc[:,-4:])

        Day  MonthNumber     Month  Year
0        14           11  November  2017
1        14           11  November  2017
2        14           11  November  2017
3        14           11  November  2017
4        14           11  November  2017
...     ...          ...       ...   ...
375937   14            6      June  2018
375938   14            6      June  2018
375939   14            6      June  2018
375940   14            6      June  2018
375941   14            6      June  2018

[373322 rows x 4 columns]


### Point 13
For each (month, country) pair, the video with the largest number of views

In [ ]:
# Find the video with the largest number of views for each month and country paired.
idx_largest_views_MC=YT_Trends.groupby(["Month","MonthNumber","country"])["views"].idxmax()

# We use the idxmax function to find the index of the video with the largest number of views for each month and country pair.
print(f"The video with the largest number of views for each month and country paired:")
YT_Trends.loc[idx_largest_views_MC,["title", "views", "MonthNumber","Month", "country"]]. \
    sort_values(["MonthNumber"]).set_index(["Month", "country"])

The video with the largest number of views for each month and country paired:


title  \
Month    country                                                      
January  CA       Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...   
         DE       Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...   
         FR       Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...   
         RU          Taylor Swift - End Game ft. Ed Sheeran, Future   
         MX       Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...   
...                                                             ...   
December MX       YouTube Rewind: The Shape of 2017 | #YouTubeRe...   
         IN       YouTube Rewind: The Shape of 2017 | #YouTubeRe...   
         GB       YouTube Rewind: The Shape of 2017 | #YouTubeRe...   
         FR       YouTube Rewind: The Shape of 2017 | #YouTubeRe...   
         DE       YouTube Rewind: The Shape of 2017 | #YouTubeRe...   

                      views  MonthNumber  
Month    country                          
January  CA        43067983            1  
         DE        37728802            1  
         FR        37728802            1  
         RU        23198594            1  
         MX        31680160            1  
...                     ...          ...  
December MX       100912384           12  
         IN       125432237           12  
         GB       169884583           12  
         FR       100911567           12  
         DE       113876217           12  

[77 rows x 3 columns]

### Point 14
Read all json files with the video categories

In [15]:
folder_path = "C:\\Users\\kekko\\Downloads\\Python\\Foundations of Computer Science\\json_Data\\"
countries = ['CA', 'DE', 'FR', 'GB', 'IN', 'JP', 'KR', 'MX', 'RU', 'US']

dataframes_list = []

# Loop through each country, read the corresponding file, and create a DataFrame. 
# Add a 'country' column to each DataFrame and append it to the list.
for country in countries:
    filename = f"{country}_category_id.json"
    full_path = folder_path + filename

    with open(full_path, 'r') as f:
        data = json.load(f)
        temp_df = pd.DataFrame(data['items'])
        temp_df['country'] = country
        dataframes_list.append(temp_df)

# Concatenate all the DataFrames in the list into a single DataFrame called df_categories.
df_categories = pd.concat(dataframes_list, ignore_index=True)
df_categories['category_title'] = df_categories['snippet'].apply(lambda x: x['title'])
unique_categories = df_categories['category_title'].unique()

print(f"The dataframe has {len(df_categories)} rows and {len(df_categories.columns)} columns")
print("\nNumber of categories for country:")
print(df_categories['country'].value_counts())
print(unique_categories)

The dataframe has 311 rows and 6 columns

Number of categories for country:
country
US    32
CA    31
DE    31
FR    31
IN    31
GB    31
JP    31
KR    31
MX    31
RU    31
Name: count, dtype: int64
['Film & Animation' 'Autos & Vehicles' 'Music' 'Pets & Animals' 'Sports'
 'Short Movies' 'Travel & Events' 'Gaming' 'Videoblogging'
 'People & Blogs' 'Comedy' 'Entertainment' 'News & Politics'
 'Howto & Style' 'Education' 'Science & Technology' 'Movies'
 'Anime/Animation' 'Action/Adventure' 'Classics' 'Documentary' 'Drama'
 'Family' 'Foreign' 'Horror' 'Sci-Fi/Fantasy' 'Thriller' 'Shorts' 'Shows'
 'Trailers' 'Nonprofits & Activism']


### Point 15
For each country, determine how many videos have a category that is not assignable.

In [ ]:
# We want to find out how many videos in YT_Trends are not assignable to any category.
df_categories['is_assignable'] = df_categories['snippet'].apply(lambda x: x.get('assignable', True))
YT_Trends['category_id'] = YT_Trends['category_id'].astype(int)
df_categories['id'] = df_categories['id'].astype(int)

# We merge the YT_Trends DataFrame with the df_categories DataFrame on the category_id and country columns. 
# We use a left join to keep all the rows from YT_Trends and add the is_assign
df_merged = YT_Trends.merge(
    df_categories[['id', 'country', 'is_assignable']], # we just choose the useful columns
    left_on=['category_id', 'country'],
    right_on=['id', 'country'],
    how='left'
)

# We check if there are any videos that do not have a matching category in df_categories, which would result in NaN values in the is_assignable column.
if df_merged['is_assignable'].isna().any():
    print("There are videos that do not have a matching category in df_categories.")

# We filter the merged DataFrame to find the videos that are not assignable (is_assignable == False) and group them by country to count how many not assignable videos there are for each country.
not_assignable_videos = df_merged[df_merged['is_assignable'] == False]
outcome = not_assignable_videos.groupby('country').size()

print("Number of not assignable videos:")
if len(outcome) == 0:
    print("None")
else:
    print(outcome)

There are videos that do not have a matching category in df_categories.
Number of not assignable videos:
country
CA    130
DE    110
FR    112
GB     20
IN    221
KR    167
MX      3
RU    195
US     57
dtype: int64
